In [1]:
import json
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
fn = "../enterprise-attack/enterprise-attack.json"
with open(fn, "r", encoding="utf-8") as f:
    stix = json.load(f)

objects = stix.get("objects", [])

Need a way to associate technique names with their attack-ids

In [3]:
tech_objs = [o for o in objects if o.get("type") == "attack-pattern"]

def technique_row(o):
    # external id e.g. T1003 usually in external_references where source_name == 'mitre-attack'
    ext_refs = o.get("external_references", [])
    mitre_ref = next((r for r in ext_refs if r.get("source_name") == "mitre-attack"), {})
    external_id = mitre_ref.get("external_id")
    # kill_chain_phases may contain tactic phase names
    kcp = o.get("kill_chain_phases") or o.get("kill_chain_phases", []) or []
    phases = [p.get("phase_name") for p in kcp if isinstance(p, dict) and p.get("phase_name")]
    platforms = o.get("x_mitre_platforms") or o.get("x-mitre-platforms") or []
    data_sources = o.get("x_mitre_data_sources") or []
    return {
        "id": o.get("id"),
        "tech_name": o.get("name"),
        "external_id": external_id,
        "description": o.get("description"),
        "platforms": platforms,
        "kill_chain_phases": phases,
        "raw": o
    }

tech_df = pd.DataFrame([technique_row(o) for o in tech_objs])

tech_id_to_name = {}
for _, row in tech_df.iterrows():
    tech_id_to_name[row['id']] = row['tech_name']


In [4]:
# Filter objects to only course-of-action
mitigation_objs = [o for o in objects if o.get("type") == "course-of-action"]

def mitigation_row(o):
    return {
        "id": o.get("id"),
        "mitigation_name": o.get("name")
    }

# Build DataFrame
mitigation_df = pd.DataFrame([mitigation_row(o) for o in mitigation_objs])

# Create the ID → name dictionary
mitigation_id_to_name = pd.Series(
    mitigation_df.mitigation_name.values, 
    index=mitigation_df.id
).to_dict()

In [5]:
rel_objs = [o for o in objects if o.get("type") == "relationship"]

rel_df = pd.DataFrame([{
    "id": o.get("id"),
    "relationship_type": o.get("relationship_type"),
    "source_ref": o.get("source_ref"),
    "target_ref": o.get("target_ref"),
    "description": o.get("description"),
    "raw": o
} for o in rel_objs])

rel_df.head()


,id,relationship_type,source_ref,target_ref,description,raw
0,relationship--00038d0e-7fc7-41c3-9055-edb4d87e...,uses,malware--6a21e3a4-5ffe-4581-af9a-6a54c7536f44,attack-pattern--707399d6-ab3e-4963-9315-d9d381...,[Explosive](https://attack.mitre.org/software...,"{'type': 'relationship', 'spec_version': '2.1'..."
1,relationship--0005fb3b-274a-4ac1-8fb2-51366fcd...,mitigates,course-of-action--21da4fd4-27ad-4e9c-b93d-0b9b...,attack-pattern--43c9bc06-715b-42db-972f-52d25c...,Consider blocking download/transfer and execut...,"{'type': 'relationship', 'spec_version': '2.1'..."
2,relationship--000aa4d0-315e-40d7-b2b6-76e91ecf...,uses,intrusion-set--01e28736-2ffc-455b-9880-ed4d140...,attack-pattern--65f2d882-3f41-4d48-8a06-29af77...,[Indrik Spider](https://attack.mitre.org/group...,"{'type': 'relationship', 'spec_version': '2.1'..."
3,relationship--00192a5f-9dc0-445a-b010-d77bd08a...,uses,malware--425771c5-48b4-4ecd-9f95-74ed3fc9da59,attack-pattern--bf176076-b789-408e-8cba-7275e8...,[SombRAT](https://attack.mitre.org/software/S0...,"{'type': 'relationship', 'spec_version': '2.1'..."
4,relationship--001ecf24-8276-40d2-ba05-2d20e5c5...,uses,malware--b7010785-699f-412f-ba49-524da6033c76,attack-pattern--132d5b37-aac5-4378-a8dc-3127b1...,[GoldFinder](https://attack.mitre.org/software...,"{'type': 'relationship', 'spec_version': '2.1'..."


Read in techniques from all 3 analyses

In [9]:
tech_names = set()

with open("correlation_matrix_techniques.txt", "r") as f:
    for line in f:
        tech_names.add(line.strip())

with open("recent_apt_techniques.txt", "r") as f:
    for line in f:
        tech_names.add(line.strip())
    
with open("geographical_techniques.txt", "r") as f:
    for line in f:
        tech_names.add(line.strip())

In [11]:
len(tech_names)

51

In [10]:
# 1️⃣ Filter rel_df for "mitigates" relationships
mitigation_rels = rel_df[rel_df["relationship_type"] == "mitigates"]

# 2️⃣ Keep only relationships targeting techniques we care about
# First, get the IDs of the techniques in tech_names
tech_ids_of_interest = {tid for tid, name in tech_id_to_name.items() if name in tech_names}

mitigation_rels = mitigation_rels[mitigation_rels["target_ref"].isin(tech_ids_of_interest)]

# 3️⃣ Map source_ref (mitigation IDs) to names
mitigation_rels["mitigation_name"] = mitigation_rels["source_ref"].map(mitigation_id_to_name)

# 4️⃣ Map target_ref (tech IDs) to technique names
mitigation_rels["tech_name"] = mitigation_rels["target_ref"].map(tech_id_to_name)

# 5️⃣ Keep only relevant columns
mitigation_results = mitigation_rels[["tech_name", "mitigation_name"]]

# 6️⃣ Optionally, group by technique and list all mitigations
tech_to_mitigations = (
    mitigation_results.groupby("tech_name")["mitigation_name"]
    .apply(list)
    .to_dict()
)

# 7️⃣ Display example
for tech, mitigations in tech_to_mitigations.items():
    print(f"{tech}:")
    for m in mitigations:
        print(f"  → {m}")

Archive via Utility:
  → Audit
Credentials from Web Browsers:
  → Update Software
  → User Account Management
  → User Training
  → Restrict Web-Based Content
  → Password Policies
Data from Local System:
  → Data Loss Prevention
Domain Account:
  → Multi-factor Authentication
  → Operating System Configuration
  → Operating System Configuration
  → Network Segmentation
  → Privileged Account Management
Domains:
  → Pre-compromise
  → Pre-compromise
Email Accounts:
  → Pre-compromise
  → Pre-compromise
Encrypted/Encoded File:
  → Antivirus/Antimalware
  → Behavior Prevention on Endpoint
Exploit Public-Facing Application:
  → Application Isolation and Sandboxing
  → Filter Network Traffic
  → Network Segmentation
  → Vulnerability Scanning
  → Privileged Account Management
  → Exploit Protection
  → Limit Access to Resource Over Network
  → Update Software
Exploitation for Client Execution:
  → Exploit Protection
  → Update Software
  → Application Isolation and Sandboxing
External Remo